# TokenCacheOps: Experimental Validation Notebook

This notebook reproduces the full experimental pipeline for the TokenCacheOps research paper.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

from src.config import ExperimentConfig
from src.simulation import SimulationEngine
from src.statistics import generate_summary_table, run_statistical_analysis
from src.visualization import generate_all_figures

## 1. Configure Experiment

In [ ]:
config = ExperimentConfig(
    num_requests=100_000,
    num_runs=30,
    random_seed=42,
    output_dir='../outputs'
)
engine = SimulationEngine(config)

## 2. Generate Dataset and Run Experiments

In [ ]:
results_df, ablation_df = engine.run_all_experiments(include_ablation=True)
results_df.to_csv('../outputs/data/experiment_results.csv', index=False)
ablation_df.to_csv('../outputs/data/ablation_results.csv', index=False)

## 3. Summary Statistics

In [ ]:
summary = generate_summary_table(results_df)
summary[['Method', 'cache_hit_ratio_mean', 'token_reduction_pct_mean', 'cost_reduction_pct_mean', 'roi_mean']]

## 4. Statistical Validation

In [ ]:
stats = run_statistical_analysis(results_df, 'cache_hit_ratio')
print(f"ANOVA F={stats['anova']['f_statistic']:.2f}, p={stats['anova']['p_value']:.2e}")
for method, t in stats['ttest_vs_reference'].items():
    d = stats['effect_sizes_vs_reference'][method]
    print(f"  vs {method}: t={t['t_statistic']:.2f}, p={t['p_value']:.2e}, Cohen's d={d:.2f}")

## 5. Generate Figures

In [ ]:
from pathlib import Path
fig_dir = Path('../outputs/figures')
generate_all_figures(results_df, ablation_df, fig_dir)
for i in range(1, 9):
  files = list(fig_dir.glob(f'figure{i}_*.png'))
  if files:
    display(Image(filename=str(files[0]), width=700))

## 6. Final Summary Table

In [ ]:
cols = ['method', 'cache_hit_ratio', 'token_reduction_pct', 'cost_reduction_pct', 'avg_latency_ms', 'roi']
results_df.groupby('method')[cols].agg(['mean', 'std', 'median']).round(3)